# 🔬 Notebook 3: Google Search — Deep Dive: Crawler, Ranking, Sharding

## 🛠️ Setup

```bash
cd 06-system-designs/google-search
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — the crawler

Key problems:
1. **Discovery**: where do we find URLs? → seeds + links extracted from crawled pages.
2. **Politeness**: don't hammer one host. Obey `robots.txt` and rate-limit per host.
3. **Dedup**: the same URL will show up thousands of times.
4. **Freshness**: some pages change hourly, others never. Use change-history to tune recrawl.

Frontier = priority queue keyed by (host, priority_score). One worker picks the next URL
that is *not* currently being fetched for that host.

In [ ]:
# Toy "frontier" that respects per-host politeness
from collections import deque, defaultdict
import time

class Frontier:
    def __init__(self, min_gap_s=0.0):
        self.q = deque()
        self.seen = set()
        self.last_fetch: dict[str, float] = defaultdict(float)
        self.min_gap_s = min_gap_s

    def add(self, url, host):
        if url in self.seen: return
        self.seen.add(url)
        self.q.append((url, host))

    def next_ready(self):
        now = time.time()
        for i, (url, host) in enumerate(self.q):
            if now - self.last_fetch[host] >= self.min_gap_s:
                self.q.remove((url, host))
                self.last_fetch[host] = now
                return url, host
        return None

f = Frontier(min_gap_s=0.01)
for i in range(5): f.add(f"https://a.com/{i}", "a.com")
for i in range(3): f.add(f"https://b.com/{i}", "b.com")

for _ in range(8):
    n = f.next_ready()
    print("fetching:", n)


## Deep dive 2 — ranking

Classic ingredients:
- **TF-IDF / BM25**: how well does the doc match query words?
- **PageRank**: how "important" is the page based on the link graph?
- **Freshness**: decay by age for time-sensitive queries.
- **User signals**: click-through rate, dwell time, pogo-sticking back to SERP.
- **Modern**: learned rankers — a gradient-boosted tree or neural model over hundreds of features.

Final score = weighted combination; weights are learned from click logs.

In [ ]:
# Mini PageRank on a 5-node graph
def pagerank(links: dict[str, list[str]], d=0.85, iters=30) -> dict[str, float]:
    nodes = set(links) | {b for out in links.values() for b in out}
    pr = {n: 1/len(nodes) for n in nodes}
    for _ in range(iters):
        new = {n: (1-d)/len(nodes) for n in nodes}
        for src, outs in links.items():
            if not outs: continue
            share = pr[src] / len(outs)
            for dst in outs:
                new[dst] += d * share
        pr = new
    return pr

links = {"A":["B","C"], "B":["C"], "C":["A"], "D":["C"], "E":["D","C"]}
for n, s in sorted(pagerank(links).items(), key=lambda x: -x[1]):
    print(f"{n}: {s:.3f}")


## Deep dive 3 — sharding the index

You **cannot** keep the whole inverted index on one machine. Two sharding strategies:

| Strategy | Query fan-out | Failure blast radius |
|---|---|---|
| **By term** (each shard owns a slice of vocabulary) | Low: only shards for query terms | A down shard loses some queries entirely |
| **By document** (each shard owns a doc-id range) | High: ask all shards | A down shard loses a slice of docs (graceful) |

Google uses **document sharding**. Query all shards, merge top-K.
With 1000 shards, network cost is dominated by the merge step.


## URL canonicalization and dedup

The same page can be referenced as `http://A.com/X`, `https://a.com/x/`, `https://a.com/x?utm=fb`.
Before adding to the frontier we **canonicalize** and check we have not crawled it.
At web scale the *seen set* is billions of URLs, so we use a **Bloom filter**
(tiny, probabilistic, may say yes to things it has never seen but never no to things it has).


In [ ]:
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode

TRACKING = {'utm_source','utm_medium','utm_campaign','fbclid','gclid'}

def canonicalize(url: str) -> str:
    s = urlsplit(url.strip())
    scheme = 'https'                                 # force https
    netloc = s.netloc.lower()
    path   = s.path or '/'
    if path != '/' and path.endswith('/'): path = path[:-1]
    # drop tracking params, sort the rest for stability
    qs = [(k,v) for k,v in parse_qsl(s.query) if k not in TRACKING]
    qs.sort()
    return urlunsplit((scheme, netloc, path, urlencode(qs), ''))

for raw in ['http://A.com/X/', 'https://a.com/x?utm_source=fb',
            'https://a.com/x?id=1&utm_campaign=x', 'https://a.com/x?id=1']:
    print(f'{raw:45s} -> {canonicalize(raw)}')


In [ ]:
# Tiny Bloom filter to show the idea (dependency-free).
import hashlib

class Bloom:
    def __init__(self, m_bits=1<<16, k_hashes=4):
        self.m, self.k = m_bits, k_hashes
        self.bits = bytearray(m_bits // 8)

    def _hashes(self, s):
        h = hashlib.md5(s.encode()).digest()
        for i in range(self.k):
            yield int.from_bytes(h[i*2:i*2+4], 'little') % self.m

    def add(self, s):
        for h in self._hashes(s):
            self.bits[h // 8] |= 1 << (h % 8)

    def __contains__(self, s):
        return all(self.bits[h // 8] & (1 << (h % 8)) for h in self._hashes(s))

seen = Bloom()
urls = [canonicalize(u) for u in
        ['http://a.com/1','https://a.com/1/','https://a.com/2','https://b.com/1']]
for u in urls:
    print(u, 'already crawled?', u in seen); seen.add(u)


## Respecting robots.txt

Crawlers *must* honor `robots.txt` or risk getting banned. One parser per host, cached.


In [ ]:
from urllib.robotparser import RobotFileParser

# In real code you'd fetch the file; here we feed it a literal string.
SAMPLE = (
    'User-agent: *\n'
    'Disallow: /private/\n'
    'Disallow: /admin/\n'
    'User-agent: BadBot\n'
    'Disallow: /\n'
)

rp = RobotFileParser()
rp.parse(SAMPLE.splitlines())
for bot, path in [('MyBot','/'), ('MyBot','/private/x'),
                  ('MyBot','/blog/post'), ('BadBot','/blog/post')]:
    print(f'{bot:7s} {path:15s} allowed? {rp.can_fetch(bot, "http://a.com"+path)}')


## Combining signals - BM25 x PageRank

Relevance is not just 'words match' - it is also 'is this a good site'.
The final ranker blends **lexical** (BM25) and **graph** (PageRank) signals,
usually learned but here a simple weighted sum.


In [ ]:
# Pretend BM25 scores and PageRanks for 4 candidate docs.
bm25_scores = {'a': 8.1, 'b': 7.4, 'c': 6.9, 'd': 2.1}
page_rank   = {'a': 0.05,'b': 0.40,'c': 0.10,'d': 0.60}

def combine(alpha=0.7):
    def score(d): return alpha*bm25_scores[d] + (1-alpha)*100*page_rank[d]  # rescale PR
    return sorted(bm25_scores, key=score, reverse=True)

print('BM25-heavy (alpha=0.9):', combine(0.9))
print('balanced   (alpha=0.7):', combine(0.7))
print('PR-heavy   (alpha=0.3):', combine(0.3))
# Tuning alpha is what A/B testing on click logs is for.


## Simulating a sharded query

We said Google uses **document sharding**: each shard owns a subset of the docs.
A query hits *every* shard in parallel; results are merged by score.

Below we fake 4 shards, run them concurrently with threads, and merge.


In [ ]:
import random, heapq
from concurrent.futures import ThreadPoolExecutor

random.seed(0)
N_SHARDS = 4

# Each shard returns its local top-K (doc_id, score) for the query.
def shard_query(shard_id, query, top_k=5):
    # Pretend we did BM25 on this shard's slice of the index.
    return [(f's{shard_id}-d{i}', round(random.random()*10, 2))
            for i in range(top_k)]

def aggregator(query, top_k=5):
    with ThreadPoolExecutor(max_workers=N_SHARDS) as pool:
        parts = list(pool.map(lambda s: shard_query(s, query, top_k),
                              range(N_SHARDS)))
    merged = heapq.nlargest(top_k, (r for p in parts for r in p), key=lambda x: x[1])
    return merged

print('global top-5:', aggregator('python'))
# Latency = max(shard_latency) + merge, NOT sum. That is why fan-out wins.


## Caching hot queries

A big fraction of queries are repeats ('weather', 'facebook login'). An LRU cache
in front of the ranker kills most of that traffic.


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=1024)
def serve(query):
    # imagine this calls aggregator() and costs 50ms
    return aggregator(query)

serve('python'); serve('python'); serve('rust')
print(serve.cache_info())   # hits, misses, size


## What we skipped (worth reading next)

- **Spell correction and query rewriting** - edit-distance or learned models.
- **Query understanding** - intent detection (news vs product vs navigation).
- **Learning-to-rank** - LambdaMART, neural rankers, dense-vector retrieval.
- **Freshness pipeline** - push-path for news; separate small hot index.
- **Personalization** - user profile as an extra ranking feature.
- **Abuse and SEO spam** - link spam, cloaking, duplicate-content clustering.
